In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

In [ ]:
CURRENT_FOLDER = Path.cwd()
ROOT_FOLDER = CURRENT_FOLDER.parent
assert (ROOT_FOLDER / 'main.py').exists(), f'not the repo root: {ROOT_FOLDER}'

PARTICIPANT = ROOT_FOLDER / 'recordings' / 'participant001_20260805_092548'


def load_session(session_dir):
    """Load one session's training table and its marker grid layout.

    Returns (df, layout). `layout` is an (n_markers, 2) int array indexed by
    marker_id, so layout[i] is marker i's (row, col) on the sensor.
    """
    df = pd.read_csv(session_dir / 'training_table.csv')

    with (session_dir / 'grid_layout.json').open() as f:
        meta = json.load(f)

    # A fallback_sort layout has no geometric meaning and is written as null.
    if meta['source'] != 'geometry':
        raise ValueError(
            f"{session_dir.name}: grid layout source is {meta['source']!r}, "
            f"not 'geometry' (reason: {meta['reason']})"
        )

    n_rows = meta['grid_shape']['n_rows']
    n_cols = meta['grid_shape']['n_cols']
    n_markers = n_rows * n_cols

    # JSON object keys are always strings, and their order is arbitrary --
    # allocate first, then assign by marker_id.
    layout = np.full((n_markers, 2), -1, dtype=int)
    for marker_id, pos in meta['layout'].items():
        layout[int(marker_id)] = [pos['row'], pos['col']]

    if (layout < 0).any():
        missing = np.flatnonzero(layout[:, 0] < 0).tolist()
        raise ValueError(f'{session_dir.name}: no position for marker ids {missing}')

    return df, layout

In [ ]:
# Converts (3667, 99) marker (dx, dy, etc.) columns to its row-col (3667, 11, 9)

def to_grid(df, prefix, layout):
    n_markers = len(layout)
    n_rows, n_cols = layout.max(axis=0) + 1

    # Generate names in marker_id order so values column i lines up with layout[i].
    cols = [f'{prefix}_{i}' for i in range(n_markers)]
    values = df[cols].to_numpy()

    out = np.full((len(df), n_rows, n_cols), np.nan)
    out[:, layout[:, 0], layout[:, 1]] = values

    assert not np.isnan(out).any(), "out contains NaN values"

    return out

In [ ]:
df, layout = load_session(PARTICIPANT / 'session1_20260805_093706')
grid = to_grid(df, 'dx', layout)

# Verify the scatter against grid_layout.json: each marker's value must land
# at the (row, col) the file assigns it.
assert grid.shape == (len(df), 11, 9), grid.shape
for marker_id, (row, col) in [(0, (0, 0)), (5, (0, 5)), (94, (10, 4)), (98, (10, 8))]:
    assert grid[0, row, col] == df[f'dx_{marker_id}'][0], marker_id

print('to_grid ok:', grid.shape)

In [ ]:
# converting dx, dy, mag, t, and trial to grid

dx = to_grid(df, 'dx', layout)
dy = to_grid(df, 'dy', layout)
mag = to_grid(df, 'mag', layout)
t = df['timestamp_ms'].to_numpy()
trial = df['trial_id'].to_numpy()

print(mag.shape)

In [ ]:
# maximum displacement over magnitude

def max_disp(mag_grid):
    return mag_grid.max(axis=(1,2))

md = max_disp(mag)
np.corrcoef(md, df['force_kg_corr'])[0,1]

In [ ]:
''' sin_theta and cos_theta for direction
    - "out" pre-allocates memory of zeros that has the same dimensions as dy_grid
    - "where" limits division by 0 '''
    
def direction(dx_grid, dy_grid, mag_grid):
    sin_theta = np.divide(dy_grid, mag_grid, out=np.zeros_like(dy_grid), where=mag_grid != 0)
    cos_theta = np.divide(dx_grid, mag_grid, out=np.zeros_like(dx_grid), where=mag_grid != 0)
    return sin_theta, cos_theta

sin_t, cos_t = direction(dx, dy, mag)

print(sin_t.shape, cos_t.shape)

In [ ]:
def strain(dx_grid, dy_grid):
    sx = np.diff(dx_grid, axis=2)
    sx = np.concatenate([sx, sx[:, :, -1:]], axis=2) # axis=2 for column

    sy = np.diff(dy_grid, axis=1)
    sy = np.concatenate([sy, sy[:, -1:, :]], axis=1) # axis=1 for row

    return sx, sy

In [ ]:
def velocity(dx_grid, dy_grid, timestamp_ms, trial_id):
    """Returns (vx, vy) in px/s, same shape as the input grids.

    NaN at the first frame of the session and at every trial boundary --
    there is no valid earlier frame to difference against, and 0 would be
    read by a model as "no movement here".
    """
    dt = np.diff(timestamp_ms) / 1000
    vx = np.diff(dx_grid, axis=0) / dt[:, None, None]
    vy = np.diff(dy_grid, axis=0) / dt[:, None, None]

    # np.diff drops one frame -- prepend NaN so frame i lines up across channels.
    pad = np.full((1,) + dx_grid.shape[1:], np.nan)
    vx = np.concatenate([pad, vx], axis=0)
    vy = np.concatenate([pad, vy], axis=0)

    # A velocity spanning a trial boundary measures the pause, not the muscle.
    boundaries = np.flatnonzero(np.diff(trial_id) != 0) + 1
    vx[boundaries] = np.nan
    vy[boundaries] = np.nan

    return vx, vy


vx, vy = velocity(dx, dy, t, trial)
print(vx.shape, ' NaN frames:', int(np.isnan(vx[:, 0, 0]).sum()))